# Score-O: максимум баллов за время

Пайплайн: **изображение → КП (VLM) + сегментация → матрица времён → оптимальный тур → визуализация**.

Вход: карта, скорость на открытом (`open_land`, м/с), бюджет времени.
Баллы КП = **первая цифра номера** (например `31` → `3`). Старт = финиш (треугольник S/F).

Детекция КП: `detect_controls_vlm` (Yandex AI Studio по умолчанию). Нужны `YC_API_KEY` и `YC_FOLDER_ID` в `.env`.


In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Не найден src/. Укажите ROOT вручную.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt

from src.orienteering import plan_course, print_plan_summary, show_course_plan
from src.routing.geo import meters_per_pixel_from_scale
from src.routing.inference import load_rgb
from src.utils.config import load_experiment_config
from src.utils.device import get_device

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Параметры

In [ ]:
# IMAGE_PATH = ROOT / "dataset" / "kurakina_dacha_2017_omaps" / "image.png"
# IMAGE_PATH = ROOT / "dataset" / "ufa_komsomolsky_omaps" / "image.png"
# IMAGE_PATH = ROOT / "dataset" / "ufa_kordon_omaps" / "image.png"  # Удомельское оз.
IMAGE_PATH = ROOT / "2400.jpg"

CONFIG = "configs/quality_segformer.yaml"
CHECKPOINT = ROOT / "checkpoints" / "quality_segformer_b4" / "best.pt"

USE_GT = not CHECKPOINT.exists()

# Скорость на открытом пространстве и бюджет времени:
OPEN_SPEED_MPS = 0.5   # м/с
TIME_HOURS = 2.0
TIME_BUDGET_S = TIME_HOURS * 3600.0

CELL_SIZE = 4

# Рельеф: time *= (1 + CONTOUR_PENALTY * доля_горизонталей в ячейке)
CONTOUR_PENALTY = 8.0

# Масштаб: на легенде «1:25000», в OCD/meta часто ошибочно 10000 → дистанции ~×2.5 занижены.
# None = брать resolution из meta.json; иначе явный пересчёт через meters_per_pixel_from_scale.
MAP_SCALE = 25000
MAP_DPI = 300
METERS_PER_PIXEL = meters_per_pixel_from_scale(MAP_SCALE, MAP_DPI)  # ≈ 2.117 м/px
# METERS_PER_PIXEL = None  # доверять meta.json

# Детекция КП: VLM (Yandex). Для OCR — detector="classical".
DETECTOR = "vlm"
DETECT_KWARGS = {
    "provider": "yandex",
    "model": "qwen3.6-35b-a3b",
    # Меньшие тайлы увеличивают 40 px кружки и подписи относительно кадра VLM.
    # Перекрытие не даёт потерять КП, разрезанные границей соседних тайлов.
    "tile_size": 768,
    "overlap": 256,
    "refine_circles": True,
    # Не отбрасывать слабое чтение номера до геометрической привязки к кольцу.
    "min_confidence": 0.20,
    # Для карты 2618×3000: окружности КП имеют диаметр около 40 px.
    "circle_diameter_px": 40.0,
    # Запас на VLM-точку, попавшую в номер рядом с окружностью.
    "circle_search_radius_px": 90.0,
    "start_search_radius_px": 90.0,
}

# Если треугольник S/F не находится — задайте вручную:
START_XY = None  # например (1200.0, 800.0)

assert IMAGE_PATH.exists(), IMAGE_PATH
print("image:", IMAGE_PATH)
print("checkpoint:", CHECKPOINT, "(exists)" if CHECKPOINT.exists() else "(нет → USE_GT)")
print("USE_GT:", USE_GT)
print("detector:", DETECTOR, DETECT_KWARGS)
print(f"open_land={OPEN_SPEED_MPS} м/с, budget={TIME_HOURS} ч ({TIME_BUDGET_S:.0f} с)")
print(f"contour_penalty={CONTOUR_PENALTY}, meters_per_pixel={METERS_PER_PIXEL}")
IMAGE_PATH = ROOT / "3000.jpg"

## 2. Планирование тура

`plan_course` детектирует КП через VLM (`detector="vlm"`), сегментирует карту, строит матрицу времён и решает Orienteering Problem.


In [ ]:
import os
# Credentials are read from environment variables or a local .env file.

cfg = load_experiment_config(CONFIG)
device = get_device()
print("device:", device)

model = None
ckpt = None
if not USE_GT:
    from src.infer_fullmap import load_model_from_checkpoint

    assert CHECKPOINT.exists(), f"Нет чекпоинта: {CHECKPOINT}"
    model = load_model_from_checkpoint(CHECKPOINT, cfg, device)
    ckpt = CHECKPOINT

plan = plan_course(
    IMAGE_PATH,
    open_land_speed_mps=OPEN_SPEED_MPS,
    time_budget_s=TIME_BUDGET_S,
    model=model,
    device=device,
    cfg=cfg,
    checkpoint=ckpt,
    use_gt=USE_GT,
    start_xy=START_XY,
    cell_size=CELL_SIZE,
    detector=DETECTOR,
    detect_kwargs=DETECT_KWARGS,
    reconstruct_legs=True,
    contour_penalty=CONTOUR_PENALTY,
    resolution_m_per_px=METERS_PER_PIXEL,
)

print(
    f"resolution={plan.meta.get('resolution_m_per_px'):.4f} м/px, "
    f"contour_penalty={plan.meta.get('contour_penalty')}"
)
print_plan_summary(plan)


## 3. Визуализация

In [ ]:
image = load_rgb(IMAGE_PATH)

fig, ax = show_course_plan(image, plan, figsize=(12, 12))
plt.tight_layout()
plt.show()

dist = plan.total_distance_m
dist_txt = f"{dist / 1000:.2f} км" if dist >= 1000 else f"{dist:.0f} м"
print(
    f"Итого: {plan.total_points} баллов, "
    f"{len(plan.selected)}/{len(plan.controls)} КП, "
    f"{dist_txt}, "
    f"{plan.total_time_min:.1f} мин из {plan.time_budget_s/60:.0f}"
)


## 4. (Опционально) Пошаговый разбор

Отдельно посмотреть детекцию КП через VLM — как в `controls_demo`.


In [ ]:
from src.controls import detect_controls_vlm, draw_controls
from src.orienteering import points_from_number

detection = detect_controls_vlm(IMAGE_PATH, **DETECT_KWARGS)
print("КП:", len(detection.controls), "start:", detection.start)
for c in detection.controls[:15]:
    print(f"  {c.number:>4} → {points_from_number(c.number)} баллов  ({c.x:.0f},{c.y:.0f})")
if len(detection.controls) > 15:
    print(f"  … ещё {len(detection.controls) - 15}")

preview = draw_controls(image, detection)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(preview)
ax.set_title(f"VLM: КП + S/F ({len(detection.controls)})")
ax.axis("off")
plt.show()


## 5. Полная анимация пайплайна

Один ролик:

1. **SegFormer** → **VLM** → **refine**
2. **Рёбра графа** — Dijkstra по местности (ускоренная волна от S/F + пара источников)
3. **DP** — выбор набора/порядка КП по матрице времён (пути уже «с рельефом»)
4. **Финальный маршрут** — проявление выбранного тура (без повторного поиска)

`REDETECT_VLM=True` — повторный Vision API для тайлов/refine.  
`False` — seg + рёбра + DP + финал по уже посчитанному `plan`.

In [ ]:
from src.controls import detect_controls_vlm_trace
from src.orienteering import animate_course_pipeline
from src.routing.cost_model import build_cost_grid
from src.routing.geo import MapMeta

OUT_DIR = ROOT / "runs" / "course_vis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# True → повторный VLM-вызов для анимации тайлов/refine; False → seg + рёбра + DP + финал
REDETECT_VLM = True
ANIM_FORMAT = "gif"  # или "mp4"
ANIM_MAX_SIDE = 768

assert plan.label is not None, "Нужен plan.label — перезапустите §2"

meta = MapMeta(resolution_m_per_px=float(plan.meta["resolution_m_per_px"]))
cost = build_cost_grid(
    plan.label,
    meta,
    speeds=plan.speeds,
    cell_size=CELL_SIZE,
    contour_penalty=CONTOUR_PENALTY,
)

vlm_trace = None
vlm_raw = None
if REDETECT_VLM:
    vlm_kw = {k: v for k, v in DETECT_KWARGS.items() if k != "refine_circles"}
    print("VLM re-detect (refine_circles=False) для анимации…")
    vlm_trace = detect_controls_vlm_trace(
        IMAGE_PATH,
        refine_circles=False,
        **vlm_kw,
    )
    vlm_raw = vlm_trace.result
    print(f"  тайлов: {len(vlm_trace.tile_events)}, КП: {len(vlm_raw.controls)}")

anim_path = animate_course_pipeline(
    image,
    plan,
    cost=cost,
    vlm_trace=vlm_trace,
    vlm_raw=vlm_raw,
    out_path=OUT_DIR / f"{IMAGE_PATH.parent.name}_pipeline.{ANIM_FORMAT}",
    format=ANIM_FORMAT,
    fps=14,
    max_side=ANIM_MAX_SIDE,
    seg_mode="scan",
    include_seg=True,
    include_vlm=REDETECT_VLM,
    include_refine=REDETECT_VLM,
    include_matrix=True,   # Dijkstra → рёбра / матрица времён
    include_solver=True,   # DP по матрице
    include_path=True,     # финальный тур (без повторных волн A*)
    n_wave_frames=22,      # ускоренный рендер волны рёбер
    animate_extra_sources=2,
    n_path_frames=14,
    hold_stage=10,
    display=True,
)

print("saved:", anim_path)
print(
    f"пайплайн · {ANIM_FORMAT} · max_side={ANIM_MAX_SIDE} · "
    f"тур: {plan.order_numbers}"
)